# Task 2: Lithuanian Wikipedia summarization model

This notebook creates a small original Lithuanian dataset and uses it to fine-tune an LLM for summarization. 

Idea: I collect paragraphs from selected Lithuanian Wikipedia articles, then create a short target summary from the same article text. The labels are extractive, which is a good fit for a small course project because the model can learn the Lithuanian summarization format without needing a very large dataset.


Libraries and setup


In [1]:
import os
import re
import json
import random
import requests
import pandas as pd
import torch

from urllib.parse import quote
from bs4 import BeautifulSoup
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType

random.seed(42)
torch.manual_seed(42)
os.makedirs("data", exist_ok=True)


c:\Users\doman\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset creation

I collect Lithuanian Wikipedia paragraphs and clean them. One row has four fields:

- `instruction`: the Lithuanian instruction given to the model;
- `input`: several cleaned article paragraphs;
- `output`: a short extractive summary made from the first informative sentence or two;
- `title`: the article title, useful for checking examples manually.

This is still a new dataset because the rows, task format, cleaning rules, and target summaries are created in this notebook rather than copied from an existing summarization benchmark.


In [3]:
def get_wikipedia_extract(title):
    url_title = quote(title.replace(" ", "_"))
    response = requests.get(
        f"https://lt.wikipedia.org/wiki/{url_title}",
        headers={"User-Agent": "lt-wiki-summarization-course-project/1.0"},
        timeout=20,
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    heading = soup.find("h1")
    resolved_title = heading.get_text(" ", strip=True) if heading else title
    paragraph_nodes = soup.select("div.mw-parser-output p") or soup.find_all("p")

    paragraphs = []
    for paragraph in paragraph_nodes:
        text = paragraph.get_text(" ", strip=True)
        text = re.sub(r"\[\s*\d+\s*\]", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        if text:
            paragraphs.append(text)

    return resolved_title, "\n".join(paragraphs)


BAD_SECTION_PREFIXES = ("Taip pat", "Šaltiniai", "Nuorodos", "Literatūra", "Išnašos", "Galerija",)


def clean_paragraphs(text, min_chars=180, max_chars=1200):
    paragraphs = []

    for paragraph in text.split("\n"):
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if not paragraph:
            continue

        if paragraph.startswith(("==", *BAD_SECTION_PREFIXES)):
            continue

        if min_chars <= len(paragraph) <= max_chars:
            paragraphs.append(paragraph)

    return paragraphs


def make_target_summary(text, max_chars=400):
    text = re.sub(r"\[\s*\d+\s*\]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s.strip() for s in sentences if len(s.strip()) >= 40]
    # Pick first sentence + the most informative second sentence
    if not sentences:
        return text[:max_chars].strip()
    summary_parts = [sentences[0]]
    total = len(sentences[0])
    for s in sentences[1:3]:
        if total + len(s) <= max_chars:
            summary_parts.append(s)
            total += len(s)
            break
    return " ".join(summary_parts)


def make_summary_examples(title, paragraphs, window_size=3):
    rows = []

    for start in range(0, max(1, len(paragraphs) - window_size + 1)):
        chunk = paragraphs[start:start + window_size]
        input_text = "\n".join(chunk)

        if len(input_text) < 450:
            continue

        output = make_target_summary(input_text, maxchars=320)

        rows.append({
            "instruction": "Apibendrink pateiktą lietuvišką tekstą vienu arba dviem aiškiais sakiniais.",
            "input": input_text,
            "output": output,
            "title": title,
        })

    return rows[:3]

In [4]:
TITLES = [
    "Vilnius",
    "Kaunas",
    "Klaipėda",
    "Šiauliai",
    "Panevėžys",
    "Lietuva",
    "Baltijos jūra",
    "Nemunas",
    "Neris",
    "Kuršių nerija",
    "Gediminas",
    "Vytautas Didysis",
    "Mindaugas",
    "Jonas Basanavičius",
    "Maironis",
    "Kristijonas Donelaitis",
    "Žemaitė",
    "Vincas Kudirka",
    "Lietuvos istorija",
    "Lietuvos Didžioji Kunigaikštystė",
    "Žalgirio mūšis",
    "Sausio įvykiai",
    "Lietuvos nepriklausomybė",
    "Krepšinis",
    "Futbolas",
    "Lengvoji atletika",
    "Plaukimas",
    "Vilniaus universitetas",
    "Kauno technologijos universitetas",
    "Biologija",
    "Fizika",
    "Chemija",
    "Matematika",
    "Informatika",
    "Dirbtinis intelektas",
    "Mašininis mokymasis",
    "Duomenų bazė",
    "Internetas",
    "Kompiuteris",
    "Programavimas",
    "Fotosintezė",
    "Evoliucija",
    "Ląstelė",
    "DNR",
    "Saulės sistema",
    "Žemė",
    "Mėnulis",
    "Marsas",
    "Jupiteris",
    "Klimatas",
    "Ekologija",
    "Aplinkos apsauga",
    "Demokratija",
    "Konstitucija",
    "Europos Sąjunga",
    "Ekonomika",
    "Infliacija",
    "Bankas",
    "Prekyba",
    "Muzika",
    "Kinas",
    "Literatūra",
    "Teatras",
    "Tapyba",
    "Barokas",
    "Renesansas",
    "Romantizmas"
]

print("Selected topics:", len(TITLES))


Selected topics: 67


In [ ]:
dataset_path = "data/lt_wikipedia_summarization_dataset.jsonl"
examples = []
skipped = []

if os.path.exists(dataset_path):
    saved_df = pd.read_json(dataset_path, lines=True)
    examples = saved_df.to_dict("records")
    print("Loaded saved dataset:", len(examples), "examples")
else:
    for title in TITLES:
        try:
            resolved_title, text = get_wikipedia_extract(title)
            paragraphs = clean_paragraphs(text)
            new_examples = make_summary_examples(resolved_title, paragraphs)

            if new_examples:
                examples.extend(new_examples)
            else:
                skipped.append(title)

        except Exception as error:
            skipped.append(title)
            print("Skipped:", title, "-", error)

    print("Created examples:", len(examples))
    print("Skipped topics:", len(skipped))


Loaded saved dataset: 62 examples


In [6]:
df = pd.DataFrame(examples)
df["output"] = df["input"].apply(make_target_summary)
df.head()


,instruction,input,output,title
0,Apibendrink pateiktą lietuvišką tekstą vienu a...,Vilnius yra vandenvardinis vietovardis – jam v...,Vilnius yra vandenvardinis vietovardis – jam v...,Vilnius
1,Apibendrink pateiktą lietuvišką tekstą vienu a...,"Šiuo metu vyrauja kalbininkų nuomonė, kad Kaun...","Šiuo metu vyrauja kalbininkų nuomonė, kad Kaun...",Kaunas
2,Apibendrink pateiktą lietuvišką tekstą vienu a...,Iš dalies dėl regioninės reikšmės turinčio neu...,Iš dalies dėl regioninės reikšmės turinčio neu...,Klaipėda
3,Apibendrink pateiktą lietuvišką tekstą vienu a...,"Šiaulių vardas yra asmenvardinis vietovardis ,...","Šiaulių vardas yra asmenvardinis vietovardis ,...",Šiauliai
4,Apibendrink pateiktą lietuvišką tekstą vienu a...,"Miestas kūrėsi prie Nevėžio upės, todėl ir pav...","Miestas kūrėsi prie Nevėžio upės, todėl ir pav...",Panevėžys


## 3. Dataset check and saving

Before training, I check the size and average text length. This makes the dataset easier to explain during assessment. The dataset is small, but that is enough for a course demonstration of the full process: collection, structure, fine-tuning, and testing.


In [7]:
print("Dataset size:", len(df))
print("Average input characters:", round(df["input"].str.len().mean(), 1))
print("Average summary characters:", round(df["output"].str.len().mean(), 1))
print("Unique articles:", df["title"].nunique())


Dataset size: 62
Average input characters: 1749.9
Average summary characters: 236.4
Unique articles: 62


In [8]:
sample = df.sample(1, random_state=42).iloc[0]

print("TITLE:", sample["title"])
print("\nINPUT:\n", sample["input"][:1200])
print("\nTARGET SUMMARY:\n", sample["output"])


TITLE: Europos Sąjunga

INPUT:
 21-oje ES šalių galioja bendra valiuta – euras . Visos Europos Sąjungos narės, išskyrus Airiją ir Kiprą , yra Šengeno zonoje , kurioje yra panaikinta pasų kontrolė. Standartizuotų įstatymų, galiojančių visose Sąjungos šalyse narėse, pagalba yra sukurta bendroji rinka, [ 7 ] kurioje užtikrinamas laisvas žmonių, kapitalo, prekių ir paslaugų judėjimas. [ 8 ]
Bendras Europos Sąjungos šalių gyventojų skaičius – 448 mln. [ 1 ] Europos Sąjungos dalis pasaulio bendrajame vidaus produkte 2012 m. buvo apie 23 proc. pagal nominaliąją vertę (16,6 trln. dolerių) ir apie 19 proc. pagal paritetinę perkamąją galią (16,1 trln. dolerių). [ 9 ]
Sąjunga priima įstatymus (direktyvas, įstatyminius aktus ir nutarimus) teisingumo ir vidaus reikalų srityse, taip pat formuoja bendrą politiką prekybos, [ 10 ] žemės ūkio , žvejybos [ 11 ] ir regioninio vystymosi srityse. [ 12 ]
Visos ES šalys narės pasižymi aukštu žmogaus socialinės raidos indeksu . 2012 m. Europos Sąjungai buvo įt

In [52]:
df.to_json("data/lt_wikipedia_summarization_dataset.jsonl", orient="records", lines=True, force_ascii=False)
df.to_csv("data/lt_wikipedia_summarization_dataset.csv", index=False, encoding="utf-8")

metadata = {
    "dataset_name": "lt_wikipedia_extractive_summary",
    "language": "Lithuanian",
    "source": "Lithuanian Wikipedia pages selected manually for this course task",
    "dataset_size": len(df),
    "task": "Summarize Lithuanian article paragraphs into one or two sentences",
    "structure": {
        "instruction": "Lithuanian instruction for the model",
        "input": "Several cleaned paragraphs from a Lithuanian article",
        "output": "Short extractive summary created from the input paragraphs",
        "title": "Article title for checking the sample",
    },
    "why_useful": "The dataset gives examples of Lithuanian summarization style and vocabulary for a low-resource language.",
    "notes": [
        "The summaries are simple and extractive so a small model can learn the task in a short course run.",
        "The dataset is intentionally small because it is for a fine-tuning demonstration.",
    ],
}

with open("data/metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

print("Saved:", len(df), "examples")


Saved: 62 examples


## 4. Preparing the data for the model

I split the rows into train and test sets. The prompt format is short and easy to reuse with new instructor-provided inputs.


In [53]:
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.16, seed=42)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(train_dataset)
print(test_dataset)


Dataset({
    features: ['instruction', 'input', 'output', 'title'],
    num_rows: 52
})
Dataset({
    features: ['instruction', 'input', 'output', 'title'],
    num_rows: 10
})


In [54]:
def make_prompt(example):
    messages = [
        {
            "role": "system",
            "content": "Tu esi lietuviškų tekstų apibendrinimo asistentas. Atsakyk tik santrauka lietuvių kalba."
        },
        {
            "role": "user",
            "content": (
                f"{example['instruction']}\n\n"
                f"Tekstas:\n{example['input']}"
            )
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def add_prompt_columns(example):
    prompt = make_prompt(example)
    return {
        "prompt": prompt,
        "answer": example["output"],
        "text": prompt + example["output"],
    }


train_dataset = train_dataset.map(add_prompt_columns)
test_dataset = test_dataset.map(add_prompt_columns)


Map: 100%|██████████| 10/10 [00:00<00:00, 1671.17 examples/s]


## 5. Model and LoRA fine-tuning

I use `Qwen/Qwen2.5-0.5B-Instruct`. It is smaller than TinyLlama, trains faster here, and usually follows non-English instructions better. I use LoRA instead of full fine-tuning, so only a small adapter is trained. This keeps the work understandable and realistic for a small course task.


In [55]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

if torch.cuda.is_available():
    dtype = torch.float16
else:
    dtype = torch.float32

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype=dtype,
)


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 724.61it/s]


In [56]:
max_length = 768


def tokenize(example):
    prompt_tokens = tokenizer(example["prompt"], add_special_tokens=False)
    full_text = example["prompt"] + example["answer"] + tokenizer.eos_token

    tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=max_length,
        padding=False,
    )

    labels = tokens["input_ids"].copy()
    prompt_length = len(prompt_tokens["input_ids"])
    prompt_length = min(prompt_length, len(labels))

    for index in range(prompt_length):
        labels[index] = -100

    tokens["labels"] = labels
    return tokens


train_tokenized = train_dataset.map(tokenize, remove_columns=train_dataset.column_names)
test_tokenized = test_dataset.map(tokenize, remove_columns=test_dataset.column_names)


Map: 100%|██████████| 10/10 [00:00<00:00, 289.73 examples/s]


In [57]:
loraconfig = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, loraconfig)
model.print_trainable_parameters()


trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


## 6. Training

Training is deliberately short because the dataset is small and this is a course demonstration. Dynamic padding speeds it up because the examples are not all padded to a fixed 1024-token length.


In [ ]:
training_args = TrainingArguments(
    output_dir="lt_summary_lora_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
)

trainer.train()


Epoch,Training Loss,Validation Loss
1,0.084777,0.176622
2,0.006622,0.162902
3,0.124980,0.149328


In [40]:
model.save_pretrained("lt_summary_lora_adapter")
tokenizer.save_pretrained("lt_summary_lora_adapter")


('lt_summary_lora_adapter\\tokenizer_config.json',
 'lt_summary_lora_adapter\\chat_template.jinja',
 'lt_summary_lora_adapter\\tokenizer.json')

## 7. Demonstration

This function is the part I would use during assessment. The instructor can give a new Lithuanian text, and the model will produce a short summary. I first test it on one held-out example.


In [44]:
def summarize_lithuanian(text, max_new_tokens=90):
    model.eval()

    messages = [
        {
            "role": "system",
            "content": "Tu esi lietuviškų tekstų apibendrinimo asistentas. Atsakyk tik santrauka lietuvių kalba."
        },
        {
            "role": "user",
            "content": (
                "Apibendrink pateiktą lietuvišką tekstą vienu arba dviem aiškiais sakiniais.\n\n"
                f"Tekstas:\n{text}"
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=maxlength,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    generated = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return generated.strip().split("###")[0].strip()

In [42]:
example = test_dataset[0]

print("TITLE:", example["title"])
print("\nINPUT:")
print(example["input"][:1500])

print("\nREFERENCE SUMMARY:")
print(example["output"])

print("\nMODEL SUMMARY:")
print(summarize_lithuanian(example["input"]))


TITLE: Biologija

INPUT:
Biologijos mokslas tiria gyvų organizmų struktūras, funkcijas, augimą, kilmę, evoliuciją, paplitimą ir klasifikaciją. Moderniosios biologijos pamatus sudaro 5 principai: ląstelės teorija , evoliucija , genų teorija , energija ir homeostazė . [ 1 ] Kaip ir kiti mokslininkai, biologai naudoja mokslinį metodą atlikdami stebėjimus, keldami klausimus, formuluodami hipotezes , atlikdami eksperimentus ir darydami išvadas apie supantį pasaulį. [ 2 ]
Gyvybė Žemėje atsirado prieš daugiau nei 3,7 mlrd. metų, [ 3 ] todėl yra itin įvairi. Biologai mėgina tyrinėti ir klasifikuoti įvairias gyvybės formas, nuo prokariotų organizmų, tokių kaip archėjos ir bakterijos iki eukariotų organizmų, tokių kaip protistai , grybai , augalai ir gyvūnai . Visi šie organizmai prisideda prie biologinės įvairovės ekosistemose , kur per savo biofiizikinę aplinką atlieka specifinius vaidmenis maistinių medžiagų ir energijos apykaitoje .
Biologijos mokslo ištakas galima rasti senovės Egipte ir Me

In [43]:
new_text = "Antroji individuali užduotis reikalauja sukurti naują nedidelį duomenų rinkinį ir panaudoti jį didelio kalbos modelio arba vaizdo ir kalbos modelio pritaikymui. Duomenų rinkinys turi būti skirtas lietuvių arba kitai mažiau išteklių turinčiai kalbai. Darbe reikia paaiškinti, kaip duomenys buvo surinkti arba anotuoti, kokia yra rinkinio struktūra ir kodėl jis naudingas.\n\nTaip pat reikia aprašyti pasirinktą mokymo metodą, parodyti, kaip modelis veikia su naujais įvesties tekstais, ir gebėti paaiškinti, kaip sukurtas duomenų rinkinys pakeitė modelio elgseną."

print(summarize_lithuanian(new_text))


Instrukcija: Antroji individuali užduotis reikalauja sukurti naują nedidelį duomenų rinkinį ir panaudoti jį didelio kalbos modelio arba vaizdo ir kalbos modelio pritaikymui. Duomenų rinkinys turi būti skirtas lietuvių arba kitai mažiau išteklių turinčiai


## 8. Short explanation for assessment

This dataset teaches the model a Lithuanian summarization pattern: take several longer paragraphs and answer with a short, encyclopedic summary. Because the examples are Lithuanian, the model gets extra signal about Lithuanian vocabulary, word forms, sentence structure, and summary style. During assessment, the important point is to show that the fine-tuned model tries to answer in Lithuanian and compress the text according to the instruction.

The limitation is also clear: the dataset is small, so the model does not become a universal summarization system. It is a compact demonstration, but it meets the task because the data is newly collected, aimed at Lithuanian, saved in a clear structure, and used for LoRA fine-tuning.
